In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Relocated from the source revision recorded in ../MIGRATION_MANIFEST.json.
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md.


# Pick up both cubes and place them in a box


This is the shared manipulation task for Articles 2 and 3: grasp the red cube, lift and carry it into the receiving box, release it, then repeat for the blue cube. The gripper withdraws and both cubes must settle.

Run the completed reference first. Then open `so101_pick_place.py` and complete Steps 0–4 to build the same CPU simulation yourself. The script already includes the box observer and physical checks. The earlier stacking exercise remains optional below.


## Select the reference and robot profiles

Use the Article 2 environment and start this notebook in `part1`. The default runs both SO-101 and reBot DevArm. Set `REFERENCE = False` after completing your exercise.


## Open this lesson

Run from the checked-out tutorial in its Python 3.12 environment. In Colab, the setup fetches the selected public source and installs the hash-locked packages only in a fresh Python 3.12 kernel. Hosted execution has not been validated. Existing source files are preserved.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/johnnynunez/accelerated-computing-hub.git"
REF = "feature/blog2-validated-box-benchmark"
TUTORIAL_RELATIVE = Path("tutorials/sim2real-blogs/notebooks/mujoco")
COLAB_CHECKOUT = Path("/content/ach-mujoco-box-source")

if IN_COLAB:
    repo_root = COLAB_CHECKOUT
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    if not repo_root.exists():
        repo_root.mkdir(parents=True)
        subprocess.run(["git", "init", str(repo_root)], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "remote", "add", "origin", REPO_URL], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "sparse-checkout", "init", "--cone"], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "sparse-checkout", "set", str(TUTORIAL_RELATIVE)], check=True, env=git_env)
    actual_remote = subprocess.check_output(["git", "-C", str(repo_root), "remote", "get-url", "origin"], text=True).strip()
    if actual_remote != REPO_URL:
        raise RuntimeError("Existing checkout uses another repository; choose a fresh COLAB_CHECKOUT.")
    subprocess.run(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", REF], check=True, env=git_env)
    requested = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "FETCH_HEAD"], text=True).strip()
    existing = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if existing.returncode == 0 and existing.stdout.strip() != requested:
        raise RuntimeError("REF changed; existing files were preserved. Choose a fresh COLAB_CHECKOUT.")
    if existing.returncode != 0:
        subprocess.run(["git", "-C", str(repo_root), "checkout", "--detach", requested], check=True, env=git_env)
else:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / TUTORIAL_RELATIVE / "requirements.lock.txt").is_file()), None)
    if repo_root is None:
        raise RuntimeError("Open the notebook inside the accelerated-computing-hub checkout.")

tutorial_dir = repo_root / TUTORIAL_RELATIVE
lesson_dir = tutorial_dir / "part1"
if not (lesson_dir / "viewer_window.py").is_file():
    raise RuntimeError("This checkout does not contain the current box lessons.")
if IN_COLAB:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("These teaching cells require a Python 3.12 kernel. Use the local/Brev environment; the benchmark notebook provides an isolated Python 3.12 runtime.")
    if any(name in sys.modules for name in ("mujoco", "mujoco_warp", "warp")):
        raise RuntimeError("Restart the runtime before installing the locked physics packages.")
    os.environ.setdefault("MUJOCO_GL", "egl")
    subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", str(tutorial_dir / "requirements.lock.txt")], check=True)
os.chdir(lesson_dir)
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
revision = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "HEAD"], capture_output=True, text=True)
print("Source:", revision.stdout.strip() if revision.returncode == 0 else "Image snapshot without Git metadata; retain source fingerprints.")
print("Lesson directory:", lesson_dir)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

PART1 = Path.cwd().resolve()
assert PART1.name == "part1" and (PART1 / "box_task.py").is_file()
REFERENCE = True  # False runs your completed TODO exercises.
ROBOTS = ("so101", "rebot")
SCRIPT = PART1 / ("solutions/so101_pick_place_solution.py" if REFERENCE else "so101_pick_place.py")
OUTPUT = PART1 / ".generated" / "box_lesson" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT.mkdir(parents=True)
print("Python:", sys.executable)
print("Reports:", OUTPUT)


## Run the complete task

The controller runs at 50 Hz. Each frame advances 20 physics substeps of 1 ms: 2,000 frames cover 40 simulated seconds, including settling. Failed grasps, incomplete transport and failed placement raise an error. This one-world lesson is a correctness check; the separate benchmark notebook measures scaling.


In [ ]:
for robot in ROBOTS:
    report = OUTPUT / f"{robot}_cpu.json"
    subprocess.run([sys.executable, str(SCRIPT), "--task", "box", "--robot", robot,
                    "--sim-substeps", "20", "--headless-steps", "2000", "--report", str(report)],
                   check=True)
    result = json.loads(report.read_text())
    assert result["success"], f"The physical box task failed: {report}"
    print(robot, "both cubes grasped, carried, released and settled")


## Build the CPU loop yourself

Complete Steps 0–4 in `so101_pick_place.py`, using the snippets in `solutions/` as needed. Then switch `REFERENCE` to `False` and rerun the box task above.


### 0. Import MuJoCo

The file imports `numpy` but not the physics engine. At `TODO Step 0`, import `mujoco` for the model, state and stepping API. The completed script uses the shared `viewer_window.py` helper to render the scene.

### 1. Compile the model and allocate state

At `TODO Step 1`, replace the two `None` placeholders:

- Build the model with `load_pick_place_model(xml_path, spec)`. Do **not** call `mujoco.MjModel.from_xml_path()` directly — the helper also applies the shared arm force boost, which keeps every part of this series simulating an identical model.
- Allocate the mutable state with `mujoco.MjData(model)`.

Recall from the previous notebook: the model is read-only and shared, the data is per-simulation and mutable. That split is what makes the GPU batching in Part 2 possible.

### 2. Set the initial conditions

At `TODO Step 2`, put the robot and props in their starting configuration using two helpers from `pick_place_common`:

- `apply_arm_ctrl(model, data, spec.home_ctrl)` — snap the arm to the grasp-ready home pose and run forward kinematics.
- `reset_cubes(model, data, spec)` — place the two free-joint cubes at their spawn poses with zero velocity.

Teleporting state like this is fine for initialization. Once the simulation loop begins, state must evolve through `mj_step` dynamics instead.

### 3. Create the controller

At `TODO Step 3`, select the controller for the chosen task:

```python
controller = box_task.make_controller() if box_task else PickPlaceController(spec=spec)
```

The notebook selects the box controller. It picks the red cube first, then the blue cube. The script initializes the arm above the first cube before physics; afterward the controller changes only actuator targets.

This object owns the waypoint state machine, the damped-least-squares IK solve, and the gradual gripper ramp you saw in the previous notebook. It is backend-agnostic, which is why Parts 2 and 3 will reuse it without modification.

### 4. Step the physics

This is the core of the task. At `TODO Step 4`, write the inner loop that advances the simulation.

For each of the `sim_substeps` iterations you need to:

1. Copy the controller output into `data.ctrl[: model.nu]`.
2. Call `mujoco.mj_step(model, data)`.

Why substeps? The controller runs at 50 Hz because that is a realistic command rate for a servo, but contact-rich physics needs a much smaller timestep to stay stable. Holding the command constant across several physics steps is exactly what a real servo does between commands.

Keep an eye on this line for the rest of the series. It is the one thing that changes between backends:

| Part | Physics call |
|------|--------------|
| 1 | `mujoco.mj_step(model, data)` |
| 2 | `mjw.step(m, d)` |
| 3 | `solver.step(state_0, state_1, control, contacts, sim_dt)` |

## Check the physical result

The shared `box_task.py` observer follows both objects through grasp, lift, carry, opening, release and settling. A cube passing through the box is insufficient: the final checks require the complete cube inside and sustained low motion. The gripper must finish above the box.

The observer is already wired into the box branch; `TODO Step 5` belongs to the optional earlier stacking exercise.


## Watch the same task

From this directory, run the completed CPU reference with a viewer:

```bash
python solutions/so101_pick_place_solution.py --task box --robot so101 --sim-substeps 20
```

The viewer closes after the full 40 simulated seconds: 2,000 control frames at the default 50 Hz. Press Space to pause or resume, Esc or Q to cancel, or Ctrl+C in the terminal to stop. Cancelling before the full episode skips the final task check. A completed box run performs the same physical checks as the headless run.

The example uses a small MuJoCo rendering window built with GLFW. On macOS, launch this custom viewer with the environment’s regular `python`, so GLFW runs on the main OS thread.


## Optional earlier stacking exercise

The original `TODO Step 5` reports the stack separation. Complete it if you want to revisit the earlier red-on-blue task. At the default 50 Hz, it runs for 600 control frames (12 simulated seconds), then prints the stack diagnostics. It is a separate exercise from the box benchmark.


### 5. Verify the stack

At `TODO Step 5`, print the final cube positions and compare the two distances below. This step reports diagnostic values; it does not assert stacking success.

Read `data.xpos[red_body]` and `data.xpos[blue_body]`, then report two numbers:

- the horizontal distance between the two cube centers (should be close to zero)
- the vertical gap between them (should be roughly `2 * 0.022 = 0.044` m, one cube height)

In [ ]:
RUN_STACK = False
if RUN_STACK:
    for robot in ROBOTS:
        subprocess.run([sys.executable, str(SCRIPT), "--task", "stack", "--robot", robot,
                        "--headless-steps", "600"], check=True)


## Next

Continue to `../part2/03__mujoco_warp.ipynb` to move this same box task onto a GPU. Use `../part2/04__cpu_gpu_benchmark.ipynb` for the measured CPU/GPU comparison across world counts; a one-world stepping loop does not predict batched throughput.
